# Random points on the surface of a unit sphere

**Goal:** generate $N$ points $\vec p=(x,y,z)$ with $x^2+y^2+z^2=1$ such that every region of the sphere receives, on average, a number of points **proportional to its area** (uniform distribution on $S^2$).

**The trap.** Choosing the polar angle $\theta\in[0,\pi]$ and the azimuth $\phi\in[0,2\pi)$ uniformly looks natural, but it is *wrong*: points pile up near the poles.

**Why.** The area element on the sphere is
$$d\Omega=\sin\theta\,d\theta\,d\phi=-d(\cos\theta)\,d\phi .$$
A uniform density on the sphere is therefore uniform in $\phi$ **and in $z=\cos\theta$** (Archimedes' hat-box theorem: equal-height bands have equal area).

**Methods we will implement**

| # | Method | Idea |
|---|--------|------|
| 1 | naive (wrong) | $\theta\sim U(0,\pi)$, $\phi\sim U(0,2\pi)$ |
| 2 | uniform $\cos\theta$ | $z\sim U(-1,1)$, $\phi\sim U(0,2\pi)$ |
| 3 | normalised Gaussian | $\vec g\sim N(0,I_3)$, $\vec p=\vec g/|\vec g|$ |
| 4 | rejection | uniform in the cube $[-1,1]^3$, keep points inside the unit ball, normalise |

Method 3 connects to the Box–Muller notebook: the Gaussian numbers can come from the Box–Muller transform.

## How to use this notebook
Complete the cells marked `# TODO` (replace `None`). Each exercise ends with automatic checks (✅ / ❌).
If you are stuck, click **Show solution** under the exercise (a collapsible text block, not code). Try on your own first!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["figure.figsize"] = (6, 4)

def check(name, cond):
    """Run a test (a function returning True/False) and print a tick or a cross."""
    try:
        ok = bool(cond())
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + name)

def plot_sphere(ax, p, title="", s=2):
    """3D scatter plot of an (N, 3) array of points on the unit sphere."""
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=s, alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1)); ax.view_init(elev=20, azim=35)

## Exercise 1 — The naive (wrong) method
Complete `sphere_naive(n, rng)`: draw $\theta\sim U(0,\pi)$ and $\phi\sim U(0,2\pi)$, then convert to Cartesian coordinates
$$x=\sin\theta\cos\phi,\quad y=\sin\theta\sin\phi,\quad z=\cos\theta .$$
The function returns an array of shape `(n, 3)`.

In [ ]:
def sphere_naive(n, rng):
    """NAIVE sampling: theta and phi uniform (NOT uniform on the sphere!)."""
    theta = None   # TODO
    phi   = None   # TODO
    x = None       # TODO
    y = None       # TODO
    z = None       # TODO
    return np.column_stack([x, y, z])


check("shape is (1000, 3)", lambda: sphere_naive(1000, np.random.default_rng(0)).shape == (1000, 3))
check("all points on the unit sphere",
      lambda: np.allclose(np.linalg.norm(sphere_naive(1000, np.random.default_rng(0)), axis=1), 1))

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
def sphere_naive(n, rng):
    """NAIVE sampling: theta and phi uniform (NOT uniform on the sphere!)."""
    theta = rng.uniform(0, np.pi, n)
    phi   = rng.uniform(0, 2 * np.pi, n)
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return np.column_stack([x, y, z])


check("shape is (1000, 3)", lambda: sphere_naive(1000, np.random.default_rng(0)).shape == (1000, 3))
check("all points on the unit sphere",
      lambda: np.allclose(np.linalg.norm(sphere_naive(1000, np.random.default_rng(0)), axis=1), 1))
```

</details>

## Exercise 2 — Uniform in $\cos\theta$
Since $d\Omega=-d(\cos\theta)\,d\phi$, draw $u=\cos\theta\sim U(-1,1)$ and $\phi\sim U(0,2\pi)$. With $\sin\theta=\sqrt{1-u^2}$:
$$x=\sqrt{1-u^2}\cos\phi,\quad y=\sqrt{1-u^2}\sin\phi,\quad z=u .$$
Complete `sphere_cos(n, rng)`.

In [ ]:
def sphere_cos(n, rng):
    """Uniform points on the sphere: z = cos(theta) uniform in [-1, 1]."""
    u   = None   # TODO: cos(theta)
    phi = None   # TODO
    s   = None   # TODO: sin(theta), expressed through u
    return np.column_stack([None, None, None])   # TODO: replace the three None


check("shape is (1000, 3)", lambda: sphere_cos(1000, np.random.default_rng(0)).shape == (1000, 3))
check("all points on the unit sphere",
      lambda: np.allclose(np.linalg.norm(sphere_cos(1000, np.random.default_rng(0)), axis=1), 1))
check("z is uniform in [-1,1] (KS p-value > 0.001)",
      lambda: stats.kstest(sphere_cos(50_000, np.random.default_rng(1))[:, 2], stats.uniform(-1, 2).cdf).pvalue > 1e-3)

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
def sphere_cos(n, rng):
    """Uniform points on the sphere: z = cos(theta) uniform in [-1, 1]."""
    u   = rng.uniform(-1, 1, n)           # cos(theta)
    phi = rng.uniform(0, 2 * np.pi, n)
    s   = np.sqrt(1 - u**2)               # sin(theta)
    return np.column_stack([s * np.cos(phi), s * np.sin(phi), u])


check("shape is (1000, 3)", lambda: sphere_cos(1000, np.random.default_rng(0)).shape == (1000, 3))
check("all points on the unit sphere",
      lambda: np.allclose(np.linalg.norm(sphere_cos(1000, np.random.default_rng(0)), axis=1), 1))
check("z is uniform in [-1,1] (KS p-value > 0.001)",
      lambda: stats.kstest(sphere_cos(50_000, np.random.default_rng(1))[:, 2], stats.uniform(-1, 2).cdf).pvalue > 1e-3)
```

**Note:** $\sin\theta\ge0$ for $\theta\in[0,\pi]$, so the positive square root is the right choice. Both $\phi\to\phi+\pi$ and $z\to -z$ are symmetries of the sample, as they should be.

</details>

## Exercise 3 — Look at the difference
The cell below shows the two samples in 3D (rotate the view by changing `elev` and `azim` in `plot_sphere`) and the histogram of $z=\cos\theta$.

Fill in the two expected densities of $z$:

- **correct sample:** $z\sim U(-1,1)$, so $f(z)=\tfrac12$;
- **naive sample:** $\theta\sim U(0,\pi)$ gives, by change of variable $z=\cos\theta$,
  $$f(z)=f_\theta\left|\frac{d\theta}{dz}\right|=\frac{1}{\pi}\frac{1}{\sin\theta}=\frac{1}{\pi\sqrt{1-z^2}} .$$

In [ ]:
rng = np.random.default_rng(3)

# 3D view
p_naive = sphere_naive(4000, rng)
p_good  = sphere_cos(4000, rng)
fig = plt.figure(figsize=(12, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d"); plot_sphere(ax1, p_naive, "naive: $\theta$, $\phi$ uniform")
ax2 = fig.add_subplot(1, 2, 2, projection="3d"); plot_sphere(ax2, p_good,  "$\cos\theta$, $\phi$ uniform")
plt.tight_layout(); plt.show()

# histogram of z
zg = np.linspace(-0.999, 0.999, 400)
pdf_naive   = None   # TODO
pdf_uniform = None   # TODO (constant, same shape as zg)

z_naive = sphere_naive(200_000, rng)[:, 2]
z_good  = sphere_cos(200_000, rng)[:, 2]
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, z, pdf, t in [(ax[0], z_naive, pdf_naive, "naive"), (ax[1], z_good, pdf_uniform, "cos(theta) uniform")]:
    a.hist(z, bins=50, density=True, alpha=0.6, label="sample")
    a.plot(zg, pdf, "r-", lw=2, label="expected density")
    a.set_title(t); a.set_xlabel("$z=\cos\theta$"); a.set_ylim(0, 1.5); a.legend()
ax[0].set_ylabel("density")
plt.tight_layout(); plt.show()

check("naive pdf at z=0 equals 1/pi", lambda: abs(np.interp(0, zg, pdf_naive) - 1/np.pi) < 1e-3)
check("uniform pdf equals 1/2",       lambda: np.allclose(pdf_uniform, 0.5))

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
rng = np.random.default_rng(3)

# 3D view
p_naive = sphere_naive(4000, rng)
p_good  = sphere_cos(4000, rng)
fig = plt.figure(figsize=(12, 5.5))
ax1 = fig.add_subplot(1, 2, 1, projection="3d"); plot_sphere(ax1, p_naive, "naive: $\\theta$, $\\phi$ uniform")
ax2 = fig.add_subplot(1, 2, 2, projection="3d"); plot_sphere(ax2, p_good,  "$\\cos\\theta$, $\\phi$ uniform")
plt.tight_layout(); plt.show()

# histogram of z
zg = np.linspace(-0.999, 0.999, 400)
pdf_naive   = 1 / (np.pi * np.sqrt(1 - zg**2))
pdf_uniform = np.full_like(zg, 0.5)

z_naive = sphere_naive(200_000, rng)[:, 2]
z_good  = sphere_cos(200_000, rng)[:, 2]
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, z, pdf, t in [(ax[0], z_naive, pdf_naive, "naive"), (ax[1], z_good, pdf_uniform, "cos(theta) uniform")]:
    a.hist(z, bins=50, density=True, alpha=0.6, label="sample")
    a.plot(zg, pdf, "r-", lw=2, label="expected density")
    a.set_title(t); a.set_xlabel("$z=\\cos\\theta$"); a.set_ylim(0, 1.5); a.legend()
ax[0].set_ylabel("density")
plt.tight_layout(); plt.show()

check("naive pdf at z=0 equals 1/pi", lambda: abs(np.interp(0, zg, pdf_naive) - 1/np.pi) < 1e-3)
check("uniform pdf equals 1/2",       lambda: np.allclose(pdf_uniform, 0.5))
```

**Expected:** in the naive sample the density of $z$ is smallest at the equator ($1/\pi\approx0.32$) and diverges at the poles, so many points crowd around $z=\pm1$ (visible as bright polar caps in the 3D plot). The $\cos\theta$-uniform sample gives the flat density $1/2$.

</details>

## Exercise 4 — Normalised Gaussian vectors
The density of a 3D standard normal vector $\vec g=(g_1,g_2,g_3)$ is
$$f(\vec g)\propto e^{-(g_1^2+g_2^2+g_3^2)/2}=e^{-|\vec g|^2/2},$$
which depends only on $|\vec g|$: it is **rotationally invariant**, so the *direction* of $\vec g$ is uniformly distributed on the sphere. Hence
$$\vec p=\frac{\vec g}{|\vec g|}$$
is uniform on $S^2$. Complete `sphere_gauss(n, rng)` (use `rng.standard_normal`; you could also use your own Box–Muller generator).

In [ ]:
def sphere_gauss(n, rng):
    """Uniform points on the sphere from normalised 3D Gaussian vectors."""
    g    = None   # TODO: array of shape (n, 3) of standard normal numbers
    norm = None   # TODO: length of each row, shape (n, 1)  (np.linalg.norm with axis and keepdims)
    return g / norm


check("shape is (1000, 3)", lambda: sphere_gauss(1000, np.random.default_rng(0)).shape == (1000, 3))
check("all points on the unit sphere",
      lambda: np.allclose(np.linalg.norm(sphere_gauss(1000, np.random.default_rng(0)), axis=1), 1))
check("z is uniform in [-1,1] (KS p-value > 0.001)",
      lambda: stats.kstest(sphere_gauss(50_000, np.random.default_rng(1))[:, 2], stats.uniform(-1, 2).cdf).pvalue > 1e-3)

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
def sphere_gauss(n, rng):
    """Uniform points on the sphere from normalised 3D Gaussian vectors."""
    g    = rng.standard_normal((n, 3))
    norm = np.linalg.norm(g, axis=1, keepdims=True)
    return g / norm


check("shape is (1000, 3)", lambda: sphere_gauss(1000, np.random.default_rng(0)).shape == (1000, 3))
check("all points on the unit sphere",
      lambda: np.allclose(np.linalg.norm(sphere_gauss(1000, np.random.default_rng(0)), axis=1), 1))
check("z is uniform in [-1,1] (KS p-value > 0.001)",
      lambda: stats.kstest(sphere_gauss(50_000, np.random.default_rng(1))[:, 2], stats.uniform(-1, 2).cdf).pvalue > 1e-3)
```

**Remark:** this method works in any dimension $d$ (just use shape `(n, d)`), whereas the $\cos\theta$ trick is specific to $d=3$. In $d=3$ it also reproduces Archimedes' theorem: the $z$-coordinate comes out uniform.

</details>

## Exercise 5 — Rejection method
1. Draw points uniformly in the cube $[-1,1]^3$.
2. Keep only those with $0<r^2=x^2+y^2+z^2\le1$ (inside the unit ball; $r^2=0$ cannot be normalised).
3. Project the accepted points on the sphere: $\vec p=\vec v/r$.

The accepted points are uniform in the ball, hence their directions are uniform on the sphere. The **acceptance probability** is the ratio of volumes, $\frac{4\pi/3}{8}=\frac{\pi}{6}\approx0.524$.

Complete the mask `ok` and the normalisation in `sphere_reject(n, rng)`. It returns the points and the measured acceptance fraction.

In [ ]:
def sphere_reject(n, rng):
    """Rejection sampling. Returns (points of shape (n, 3), acceptance fraction)."""
    chunks, n_have, n_tried = [], 0, 0
    while n_have < n:
        v  = rng.uniform(-1, 1, (2 * n, 3))      # points in the cube
        r2 = np.sum(v**2, axis=1)                # squared distance from the origin
        ok = None                                # TODO: boolean mask, True for 0 < r2 <= 1
        chunks.append(None)                      # TODO: accepted points, projected on the sphere
        n_have  += ok.sum()
        n_tried += len(v)
    return np.vstack(chunks)[:n], n_have / n_tried


pts, acc = sphere_reject(100_000, np.random.default_rng(4))
print(f"acceptance = {acc:.4f}   (expected pi/6 = {np.pi/6:.4f})")
check("shape is (100000, 3)", lambda: pts.shape == (100_000, 3))
check("all points on the unit sphere", lambda: np.allclose(np.linalg.norm(pts, axis=1), 1))
check("acceptance ~ pi/6", lambda: abs(acc - np.pi/6) < 0.01)

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
def sphere_reject(n, rng):
    """Rejection sampling. Returns (points of shape (n, 3), acceptance fraction)."""
    chunks, n_have, n_tried = [], 0, 0
    while n_have < n:
        v  = rng.uniform(-1, 1, (2 * n, 3))      # points in the cube
        r2 = np.sum(v**2, axis=1)                # squared distance from the origin
        ok = (r2 > 0) & (r2 <= 1)                # inside the unit ball (and not at the origin)
        chunks.append(v[ok] / np.sqrt(r2[ok])[:, None])   # project on the sphere
        n_have  += ok.sum()
        n_tried += len(v)
    return np.vstack(chunks)[:n], n_have / n_tried


pts, acc = sphere_reject(100_000, np.random.default_rng(4))
print(f"acceptance = {acc:.4f}   (expected pi/6 = {np.pi/6:.4f})")
check("shape is (100000, 3)", lambda: pts.shape == (100_000, 3))
check("all points on the unit sphere", lambda: np.allclose(np.linalg.norm(pts, axis=1), 1))
check("acceptance ~ pi/6", lambda: abs(acc - np.pi/6) < 0.01)
```

**Note:** about 48% of the cube points are discarded, which is cheap here. In high dimension the acceptance of the cube/ball method collapses (the ball occupies a vanishing fraction of the cube), which is why the Gaussian method is preferred there.

</details>

## Exercise 6 — A quantitative uniformity test
Pictures can mislead. Split the sphere into `nbands` bands of equal width in $z$: by Archimedes' theorem they have **equal area**, so a uniform sample should put $N/\text{nbands}$ points in each band. Use a $\chi^2$ test on the counts (`scipy.stats.chisquare` assumes equal expected counts by default).

Complete `uniformity_pvalue`, then run it on the four methods.

In [ ]:
def uniformity_pvalue(p, nbands=10):
    """chi2 p-value for 'equal counts in nbands equal-area bands in z'."""
    counts = None   # TODO: histogram of the z coordinate, nbands bins in [-1, 1]  (np.histogram returns (counts, edges))
    res    = None   # TODO: chi-square test of the counts
    return res.pvalue


rng = np.random.default_rng(100)
n = 100_000
samples = {
    "naive":      sphere_naive(n, rng),
    "cos(theta)": sphere_cos(n, rng),
    "gaussian":   sphere_gauss(n, rng),
    "rejection":  sphere_reject(n, rng)[0],
}
for name, p in samples.items():
    print(f"{name:12s} p-value = {uniformity_pvalue(p):.3g}")

check("naive method is rejected (p < 0.001)", lambda: uniformity_pvalue(samples["naive"]) < 1e-3)
check("the other three are compatible with uniform (p > 0.001)",
      lambda: all(uniformity_pvalue(p) > 1e-3 for k, p in samples.items() if k != "naive"))

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
def uniformity_pvalue(p, nbands=10):
    """chi2 p-value for 'equal counts in nbands equal-area bands in z'."""
    counts, _ = np.histogram(p[:, 2], bins=nbands, range=(-1, 1))
    res = stats.chisquare(counts)
    return res.pvalue


rng = np.random.default_rng(100)
n = 100_000
samples = {
    "naive":      sphere_naive(n, rng),
    "cos(theta)": sphere_cos(n, rng),
    "gaussian":   sphere_gauss(n, rng),
    "rejection":  sphere_reject(n, rng)[0],
}
for name, p in samples.items():
    print(f"{name:12s} p-value = {uniformity_pvalue(p):.3g}")

check("naive method is rejected (p < 0.001)", lambda: uniformity_pvalue(samples["naive"]) < 1e-3)
check("the other three are compatible with uniform (p > 0.001)",
      lambda: all(uniformity_pvalue(p) > 1e-3 for k, p in samples.items() if k != "naive"))
```

**Interpretation:** the naive sample is rejected with a tiny p-value, the three correct methods are compatible with uniformity. Remember that a large p-value is only absence of evidence against uniformity (and this test looks at $z$ only; the $\phi$ distribution could be tested the same way with bins in $\arctan2(y,x)$).

</details>

## Exercise 7 (bonus) — A Monte Carlo application
Use a uniform sample (`sphere_gauss`, $N=200\,000$) to estimate, **with uncertainties**:

1. the fraction of the sphere with $z>0.5$. The cap area is $2\pi(1-h)$ with $h=0.5$, so the expected fraction is $(1-h)/2=0.25$. Uncertainty: binomial, $\sqrt{\hat f(1-\hat f)/N}$.
2. the average $\langle x^2\rangle$ over the sphere. By symmetry $\langle x^2\rangle=\langle y^2\rangle=\langle z^2\rangle$ and their sum is 1, so the expected value is $1/3$. Uncertainty: standard error of the mean, $s/\sqrt N$.

Are both estimates within $4\sigma$ of the exact value?

In [ ]:
rng = np.random.default_rng(2026)
n = 200_000
p = sphere_gauss(n, rng)

frac = None   # TODO: fraction of points with z > 0.5
err  = None   # TODO: binomial uncertainty on frac

x2      = p[:, 0]**2
mean_x2 = None   # TODO
err_x2  = None   # TODO: standard error of the mean

print(f"fraction with z>0.5 = {frac:.4f} +- {err:.4f}   (exact 0.25)")
print(f"<x^2>               = {mean_x2:.4f} +- {err_x2:.4f}   (exact 1/3)")
check("fraction within 4 sigma", lambda: abs(frac - 0.25) < 4 * err)
check("<x^2> within 4 sigma",    lambda: abs(mean_x2 - 1/3) < 4 * err_x2)

<details>
<summary style="cursor:pointer"><b>&#9654; Show solution</b> (click to open / close)</summary>

**Solution**

```python
rng = np.random.default_rng(2026)
n = 200_000
p = sphere_gauss(n, rng)

frac = np.mean(p[:, 2] > 0.5)
err  = np.sqrt(frac * (1 - frac) / n)

x2      = p[:, 0]**2
mean_x2 = x2.mean()
err_x2  = x2.std(ddof=1) / np.sqrt(n)

print(f"fraction with z>0.5 = {frac:.4f} +- {err:.4f}   (exact 0.25)")
print(f"<x^2>               = {mean_x2:.4f} +- {err_x2:.4f}   (exact 1/3)")
check("fraction within 4 sigma", lambda: abs(frac - 0.25) < 4 * err)
check("<x^2> within 4 sigma",    lambda: abs(mean_x2 - 1/3) < 4 * err_x2)
```

**Note:** the statistical uncertainty decreases as $1/\sqrt N$: to gain one more digit you need 100 times more points.

</details>

## Questions to think about
1. Why does the naive method put too many points near the poles? Relate your answer to the area of the band between $\theta$ and $\theta+d\theta$.
2. How would you generate uniform points on a *circle*? On a $d$-dimensional hypersphere?
3. How would you generate points uniformly inside the *ball* (not on the surface)? Hint: what is the distribution of $r$ if the density is uniform in volume?